In [88]:
# after vector similairty search
condense_length_1 = 8000 # chars

# after reranker
condense_length_2 = 4000 # chars

In [89]:
from dotenv import load_dotenv
import os
import sys

load_dotenv()
# set the python path
sys.path.append(os.getenv('PYTHON_PATH'))

In [90]:
from s3 import download_model_list_from_s3
from langchain.docstore.document import Document
from format import format_topic
from langchain_openai import OpenAIEmbeddings
from portkey_ai import PORTKEY_GATEWAY_URL, createHeaders, AsyncPortkey
import lancedb
import pandas as pd
import cohere  
from portkey_ai import Portkey

In [91]:
def to_single_string(documents):
        return "".join([
            doc.page_content + "\n\n"
            for doc in documents
        ]).strip()

In [92]:
# test data
topics = [
    {
        "presentation_id": "018efa53-2169-7dc0-ad63-e7084a09afc4",
        "presentation_topic": "Exploring Cultural Contrasts: Japan vs. the USA - Insights for American Expats",
        "presentation_audience": "American expats living in Japan",
        "presentation_goal": "To inform and offer practical tips to American expats living in Japan about the cultural contrasts between Japan and the USA",
        "id": "066236b0-e1e0-77d4-8000-590e777d7d29",
        "name": "Cultural Influence on Help Seeking and Support from Professionals and Others"
    },
    {
        "presentation_id": "018efa53-2169-7dc0-ad63-e7084a09afc4",
        "presentation_topic": "Exploring Cultural Contrasts: Japan vs. the USA - Insights for American Expats",
        "presentation_audience": "American expats living in Japan",
        "presentation_goal": "To inform and offer practical tips to American expats living in Japan about the cultural contrasts between Japan and the USA",
        "id": "066236b0-e1e0-7967-8000-39e03b240974",
        "name": "Indirect Communication in Japanese Culture"
    },
    {
        "presentation_id": "018efa6a-2208-79d3-a905-958f91b06da7",
        "presentation_topic": "Antitrust pressure against Apple: History, Antitrust Arguments, and Apple's Counterarguments",
        "presentation_audience": "Tech enthusiasts and consumers",
        "presentation_goal": "Just inform the audience without a persuasive bias",
        "id": "06623708-33af-7e71-8000-883e8038cf64",
        "name": "Apple's Antitrust Allegations and Competition Concerns"
    },
    {
        "presentation_id": "018efa76-2369-7861-b1b7-592b3e5c275f",
        "presentation_topic": "The 7 Natural Wonders of the World",
        "presentation_audience": "Travelers and nature enthusiasts",
        "presentation_goal": "To inform the audience about the 7 natural wonders of the world, their locations, and what makes them special",
        "id": "0662373e-4a90-7df1-8000-eaae8e2f7505",
        "name": "Mount Everest - The Tallest Peak in the World"
    },
    {
        "presentation_id": "018efa76-2369-7861-b1b7-592b3e5c275f",
        "presentation_topic": "The 7 Natural Wonders of the World",
        "presentation_audience": "Travelers and nature enthusiasts",
        "presentation_goal": "To inform the audience about the 7 natural wonders of the world, their locations, and what makes them special",
        "id": "06623734-5459-7ea5-8000-1842ed1b986e",
        "name": "Youngest Volcano in Northern Hemisphere - Paricutin, Mexico"
    },
    {
        "presentation_id": "018efa80-4216-71c9-b37c-b55cb8e17a2d",
        "presentation_topic": "Understanding the Transformer Model Architecture: Revolutionizing Natural Language Processing\n\nTo give an informative high-level overview of the Transformer model architecture.",
        "presentation_audience": "Artificial Intelligence and Machine Learning researchers and students",
        "presentation_goal": "To give an informative high-level overview of the Transformer model architecture.",
        "id": "0662375b-e9fe-7154-8000-0e67ffd84570",
        "name": "Multi-Head Attention Mechanisms in Transformer Architecture"
    }
]

In [93]:
topic = topics[5]

In [94]:
# Get chunked and labeled documents from S3 for whole presentation
all_docs = await download_model_list_from_s3(Document, f"{topic['presentation_id']}/all_sources_chunked_docs_with_topics.json")

#Get topic docs
topic_docs = [doc for doc in all_docs if 'topic' in doc.metadata and doc.metadata['topic']['name'] == topic['name']]
print(f"Length of original topic docs in chars: {len(to_single_string(topic_docs))}")
print(f"Original number of sources: {len(set(doc.metadata['source']['id'] for doc in topic_docs))}")

Length of original topic docs in chars: 40195
Original number of sources: 7


In [95]:
markdown = format_topic(topic_docs)
# make folder if it doesnt exist
if not os.path.exists(topic['name'].replace(' ', '_')):
    os.makedirs(topic['name'].replace(' ', '_'))
with open(f"{topic['name'].replace(' ', '_')}/whole_topic.md", 'w') as f:
    f.write(markdown)

In [96]:
# reduce by similarity search
# vectorize
portkeyHeaders = createHeaders(
    provider="openai",
    api_key=os.getenv("PORTKEY_API_KEY"),
)
openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512, base_url=PORTKEY_GATEWAY_URL, default_headers=portkeyHeaders)
vectors = await openai_embeddings.aembed_documents([doc.page_content for doc in topic_docs])
uri = "data/sample-lancedb"
async_db = await lancedb.connect_async(uri)
await async_db.drop_table("my_table2")

data = [{"vector": vector, "text": doc.page_content, "metadata": doc.metadata} for vector, doc in zip(vectors, topic_docs)]
async_tbl = await async_db.create_table("my_table2", data=data)

topic_name_embedding = await openai_embeddings.aembed_query(topic['name'])

similarity_reduced_documents = await async_tbl.vector_search(topic_name_embedding).limit(len(topic_docs)).to_pandas()
# turn dfs into list of docs
similarity_reduced_docs = [Document(page_content=row['text'], metadata={"distance": row["_distance"], **row["metadata"]}) for index, row in similarity_reduced_documents.iterrows()]



Length of topic docs after reduction by similarity: 22767
Number of sources after reduction by similarity: 7


In [ ]:
similarity_reduced_docs

In [ ]:
# minimum probability of all the docs to start with
min_prob = min([doc.metadata['topic']['probability'] for doc in similarity_reduced_docs])
# incrementally reduce by raising the minimum probability by 0.05 until we get to probability 1 or the desired length
while len(to_single_string(similarity_reduced_docs)) > condense_length_1 and min_prob < 1:
    min_prob += 0.01
    min_prob = min(min_prob, 1) # cant go over one
    # filter out docs with that probability
    similarity_reduced_docs = [doc for doc in similarity_reduced_docs if doc.metadata['distance'] >= min_prob]
    
print(f"Length of topic docs after reduction by similarity: {len(to_single_string(similarity_reduced_docs))}")
print(f"Number of sources after reduction by similarity: {len(set(doc.metadata['source']['id'] for doc in similarity_reduced_docs))}")

markdown = format_topic(similarity_reduced_docs)
# make folder if it doesnt exist
if not os.path.exists(topic['name'].replace(' ', '_')):
    os.makedirs(topic['name'].replace(' ', '_'))
with open(f"{topic['name'].replace(' ', '_')}/similarity_reduced_topic.md", 'w') as f:
    f.write(markdown)
    
topic_docs = similarity_reduced_docs

In [97]:
print(format_topic(topic_docs))

#### Source: tmpt43elso2
URL: file:///var/folders/jp/99fbbz9959d892jf52cv2c3h0000gn/T/tmpt43elso2

_from: top level_
We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This masking, combined with fact that the output embeddings are offset by one position, ensures that the predictions for position i can depend only on the known outputs at positions less than i .

Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [4, 27, 28, 22].

Motivating our use of self-attention we consider three desiderata.  One is the total computational complexity per layer. Another is the amount of computation that ca

In [98]:
# Reduce further by reranker
reranker_reduced_docs = topic_docs.copy()


In [99]:
# Show the percentage of the documents that are over 512 cohere tokens
co = cohere.Client()

# count query tokens
response = co.tokenize(text=f"What can be said about {topic['name']}?", model="rerank-english-v2.0", offline=False)
query_tokens = len(response.tokens)

for doc in reranker_reduced_docs:
    response = co.tokenize(text=doc.page_content, model="rerank-english-v2.0", offline=False)
    if len(response.tokens) + query_tokens > 512:
        print(f"Document {doc.metadata['source']['title']} has {len(response.tokens)} tokens")

In [101]:
# Initialize the Portkey client
cohere_portkey = Portkey(
    api_key=os.getenv("PORTKEY_API_KEY"), 
    virtual_key="trial-cohere-679b13" 
)

completion = cohere_portkey.post(
    '/rerank',
    return_documents=False,
    max_chunks_per_doc=1,
    model="rerank-english-v2.0",
    query=f"What can be said about {topic['name']}?",
    documents=[doc.page_content for doc in reranker_reduced_docs]
)

for result in completion.results:
    reranker_reduced_docs[result["index"]].metadata["rerank_score"] = result["relevance_score"]



In [104]:
# reduce further using reranker
# minimum probability of all the docs to start with
min_prob = min([doc.metadata['rerank_score'] for doc in reranker_reduced_docs])
# incrementally reduce by raising the minimum probability by 0.05 until we get to probability 1 or the desired length
while len(to_single_string(reranker_reduced_docs)) > condense_length_2 and min_prob < 1:
    min_prob += 0.01
    min_prob = min(min_prob, 1) # cant go over one
    # filter out docs with that probability
    reranker_reduced_docs = [doc for doc in reranker_reduced_docs if doc.metadata['rerank_score'] >= min_prob]
    
print(f"Length of topic docs after reducing further with reranker: {len(to_single_string(reranker_reduced_docs))}")
print(f"Number of sources after reducing further with reranker: {len(set(doc.metadata['source']['id'] for doc in reranker_reduced_docs))}")

markdown = format_topic(reranker_reduced_docs)
# make folder if it doesnt exist
if not os.path.exists(topic['name'].replace(' ', '_')):
    os.makedirs(topic['name'].replace(' ', '_'))
with open(f"{topic['name'].replace(' ', '_')}/reranker_further_reduced_topic.md", 'w') as f:
    f.write(markdown)

Length of topic docs after reducing further with reranker: 3902
Number of sources after reducing further with reranker: 4


In [106]:
# reduce further by similarity 
similarity_further_reduced_docs = topic_docs.copy()

min_prob = min([doc.metadata['rerank_score'] for doc in similarity_further_reduced_docs])
# incrementally reduce by raising the minimum probability by 0.05 until we get to probability 1 or the desired length
while len(to_single_string(similarity_further_reduced_docs)) > condense_length_2 and min_prob < 1:
    min_prob += 0.01
    min_prob = min(min_prob, 1) # cant go over one
    # filter out docs with that probability
    similarity_further_reduced_docs = [doc for doc in similarity_further_reduced_docs if doc.metadata['distance'] >= min_prob]
    
print(f"Length of topic docs after reducing further with similarity: {len(to_single_string(similarity_further_reduced_docs))}")
print(f"Number of sources after reducing further with similarity: {len(set(doc.metadata['source']['id'] for doc in similarity_further_reduced_docs))}")

markdown = format_topic(similarity_further_reduced_docs)
# make folder if it doesnt exist
if not os.path.exists(topic['name'].replace(' ', '_')):
    os.makedirs(topic['name'].replace(' ', '_'))
with open(f"{topic['name'].replace(' ', '_')}/similarity_further_reduced_docs.md", 'w') as f:
    f.write(markdown)

Length of topic docs after reducing further with similarity: 22767
Number of sources after reducing further with similarity: 7


In [107]:
similarity_further_reduced_docs

[Document(page_content='We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This masking, combined with fact that the output embeddings are offset by one position, ensures that the predictions for position i can depend only on the known outputs at positions less than i .', metadata={'distance': 1.0022752285003662, 'headers': {'1': None, '2': None, '3': None, '4': None}, 'source': {'download_method': None, 'extraction_method': 'openparse', 'id': '018efa82-44b9-78ec-8b7a-2545dc1a0685', 'title': 'tmpt43elso2', 'type': '/var/folders/jp/99fbbz9959d892jf52cv2c3h0000gn/T/tmpt43elso2', 'url': 'file:///var/folders/jp/99fbbz9959d892jf52cv2c3h0000gn/T/tmpt43elso2'}, 'topic': {'name': 'Multi-Head Attention Mechanisms in Transformer Architecture', 'number': 1, 'probability': 1.0}, 'rerank_score': 0.2902762}),
 Document(page_content='Self-attention, sometimes called intra-attention is an attention mechanism relating different 